# 02 — Stage 1 YOLO training

**One job:** build the detector dataset, record recipes, and optionally train one detector

Run from top to bottom. Slow actions are off by default.

In [ ]:
%pip install -q ultralytics pycocotools segmentation-models-pytorch

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json
import sys
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_DIR = Path("/content/drive/MyDrive/Google Colab/Solar Filament Segmentation Challenge")
sys.path.insert(0, str(PROJECT_DIR))
%load_ext autoreload
%autoreload 2

from src.config import DEFAULT_PIPELINE_SETTINGS, TRAINING_SETTINGS
from src.data import copy_raw_data_to_local, load_records, make_split, verify_or_write_split

LOCAL_DATA_DIR = copy_raw_data_to_local(PROJECT_DIR / "data/raw")
TRAIN_IMAGES_DIR = LOCAL_DATA_DIR / "train/train_images"
TEST_IMAGES_DIR = LOCAL_DATA_DIR / "test/test_images"
ANNOTATIONS_PATH = LOCAL_DATA_DIR / "train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"
SPLIT_MANIFEST_PATH = PROJECT_DIR / "data/processed/split_manifest.json"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

In [ ]:
BUILD_DATASET = False  # Copies 1,154 annotator-specific samples into /content
TRAIN = False
MODEL_NAME = "yolo26x.pt"
IMAGE_SIZE = 1280
FLIP_UP_DOWN = 0.5  # The next planned generalization experiment

## 1. Build the YOLO dataset from the frozen split

In [ ]:
from src.data import build_yolo_dataset

coco, records = load_records(ANNOTATIONS_PATH, TRAIN_IMAGES_DIR)
manifest, _, _ = verify_or_write_split(records, SPLIT_MANIFEST_PATH)
YOLO_DATASET_DIR = Path("/content/yolo_dataset")
if BUILD_DATASET:
    counts = build_yolo_dataset(coco, TRAIN_IMAGES_DIR, manifest, YOLO_DATASET_DIR)
    print(counts)
else:
    print("Dataset build skipped. Set BUILD_DATASET = True for training.")

## 2. Existing detector recipes

| Run | Image size | Mosaic | Best epoch | Precision | Recall | mAP50 | mAP50-95 |
|---|---:|---:|---:|---:|---:|---:|---:|
| YOLO26s | 1024 | 1.0 | — | 0.584 | 0.603 | 0.628 | 0.370 |
| YOLO26l | 1280 | 1.0 | 92 | 0.612 | 0.637 | 0.648 | 0.386 |
| YOLO26x | 1280 | 0.5 | 82 | 0.609 | 0.649 | 0.659 | 0.388 |

All existing runs used `fliplr=0.5`, `flipud=0.0`, and `degrees=0`. Their complete `args.yaml` files remain beside their weights.

### 2.1 Flip-up/down retrain

**Question:** does adding the missing vertical symmetry improve generalization?

**Varies:** `flipud=0.5`.

**Fixed:** YOLO26x, 1280, mosaic 0.5, all other saved recipe settings.

**Cost:** about 1.7 hours on A100.

**Result:** untested.

**Decision:** pending.

In [ ]:
if TRAIN:
    from ultralytics import YOLO

    assert (YOLO_DATASET_DIR / "data.yaml").exists(), "Set BUILD_DATASET = True first"
    run_name = datetime.now().strftime("%Y%m%d_%H%M%S_yolo26x_1280_flipud")
    detector = YOLO(MODEL_NAME)
    detector.train(data=str(YOLO_DATASET_DIR / "data.yaml"), epochs=300, patience=30, imgsz=IMAGE_SIZE,
                   mosaic=0.5, fliplr=0.5, flipud=FLIP_UP_DOWN, degrees=0, batch=-1, device=0,
                   project=str(PROJECT_DIR / "models/stage1"), name=run_name)
else:
    print("Training skipped.")

## 3. Training-curve comparison

In [ ]:
result_files = sorted((PROJECT_DIR / "models/stage1").glob("*/results.csv"))
for path in result_files:
    frame = pd.read_csv(path)
    metric = next((column for column in frame if "mAP50-95" in column), None)
    if metric:
        plt.plot(frame[metric], label=path.parent.name)
if result_files:
    plt.xlabel("Epoch"); plt.ylabel("Box mAP50-95"); plt.legend(); plt.show()
else:
    print("No migrated results.csv files found yet.")

## 4. Detector validation summary

In [ ]:
detector_summary = pd.DataFrame([
    {"detector": "yolo26s", "imgsz": 1024, "precision": 0.584, "recall": 0.603, "map50": 0.628, "map50_95": 0.370},
    {"detector": "yolo26l", "imgsz": 1280, "precision": 0.612, "recall": 0.637, "map50": 0.648, "map50_95": 0.386},
    {"detector": "yolo26x", "imgsz": 1280, "precision": 0.609, "recall": 0.649, "map50": 0.659, "map50_95": 0.388},
])
display(detector_summary)